# TPC-DS Retail Analytics

**Dataset:** `samples.tpcds_sf1` — store_sales, date_dim, item, store, customer  
**Difficulty:** Hard  
**Topics:** Star schema joins, YoY growth, seasonal patterns, window functions, promotion analysis


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

store_sales = spark.read.table("samples.tpcds_sf1.store_sales")
date_dim    = spark.read.table("samples.tpcds_sf1.date_dim")
item        = spark.read.table("samples.tpcds_sf1.item")
store       = spark.read.table("samples.tpcds_sf1.store")
customer    = spark.read.table("samples.tpcds_sf1.customer")

print("Tables loaded.")


## Problem 1 — Year-over-Year Revenue Growth by Item Category

Join `store_sales` + `date_dim` + `item`. Compute `total_revenue` (sum of `ss_net_paid`) per category per year.  
Use a lag window to get the prior year revenue (`prev_year_revenue`) and compute `yoy_growth_pct = (total_revenue - prev_year_revenue) / prev_year_revenue * 100`.

**Expected columns:** `i_category`, `d_year`, `total_revenue`, `prev_year_revenue`, `yoy_growth_pct`


In [ ]:
# Problem 1 — write your solution here
# Assign result to: result_1

result_1 = None  # replace this


In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
for expected in ['i_category', 'd_year', 'total_revenue', 'prev_year_revenue', 'yoy_growth_pct']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_1.count()
assert cnt > 0, f"Got 0 rows"
null_prev = result_1.filter(F.col('prev_year_revenue').isNull()).count()
assert null_prev > 0, "Expected null prev_year_revenue rows for first year of each category"
assert result_1.filter(F.col('i_category').isNull()).count() == 0, "Null categories found"
print(f"Problem 1 passed ✓  ({cnt} rows)")


## Problem 2 — Holiday vs Non-Holiday Sales Comparison

Join `store_sales` + `date_dim`. Compute `total_revenue`, `avg_net_paid`, `transaction_count`  
grouped by whether `d_holiday = 'Y'` (Holiday) or `'N'` (Non-Holiday). Express as `is_holiday` string label.

**Expected columns:** `is_holiday`, `total_revenue`, `avg_net_paid`, `transaction_count`


In [ ]:
# Problem 2 — write your solution here
# Assign result to: result_2

result_2 = None  # replace this


In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
for expected in ['is_holiday', 'total_revenue', 'avg_net_paid', 'transaction_count']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_2.count()
assert cnt == 2, f"Expected exactly 2 rows, got {cnt}"
labels = {r['is_holiday'] for r in result_2.select('is_holiday').collect()}
assert labels == {'Holiday', 'Non-Holiday'}, f"Unexpected labels: {labels}"
assert result_2.filter(F.col('transaction_count') > 0).count() == 2, "All groups must have transactions"
print(f"Problem 2 passed ✓  ({cnt} rows)")


## Problem 3 — Store Performance Ranking

Join `store_sales` + `store` on `ss_store_sk = s_store_sk`. Compute `total_net_paid`, `avg_margin_pct`  
(avg of `(ss_net_paid - ss_wholesale_cost) / ss_net_paid * 100` where `ss_net_paid > 0`),  
and `revenue_rank` using `dense_rank()` ordered by `total_net_paid` desc.

**Expected columns:** `s_store_name`, `s_city`, `s_state`, `total_net_paid`, `avg_margin_pct`, `revenue_rank`


In [ ]:
# Problem 3 — write your solution here
# Assign result to: result_3

result_3 = None  # replace this


In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
for expected in ['s_store_name', 's_city', 's_state', 'total_net_paid', 'avg_margin_pct', 'revenue_rank']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_3.count()
assert cnt > 0, f"Got 0 rows"
assert result_3.filter(F.col('revenue_rank') == 1).count() >= 1, "No rank-1 store found"
assert result_3.filter(F.col('total_net_paid') < 0).count() == 0, "Negative total_net_paid found"
print(f"Problem 3 passed ✓  ({cnt} rows)")


## Problem 4 — Promotion Effectiveness

Compare transactions where `ss_promo_sk IS NOT NULL` (promo) vs NULL (non-promo).  
Join `store_sales` + `item`. Per `i_category` compute: `promo_revenue`, `non_promo_revenue`,  
`promo_avg_price`, `non_promo_avg_price` using conditional aggregation.

**Expected columns:** `i_category`, `promo_revenue`, `non_promo_revenue`, `promo_avg_price`, `non_promo_avg_price`


In [ ]:
# Problem 4 — write your solution here
# Assign result to: result_4

result_4 = None  # replace this


In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
for expected in ['i_category', 'promo_revenue', 'non_promo_revenue', 'promo_avg_price', 'non_promo_avg_price']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_4.count()
assert cnt > 0, f"Got 0 rows"
assert result_4.filter(F.col('promo_revenue') > 0).count() > 0, "No promo revenue found"
assert result_4.filter(F.col('non_promo_revenue') > 0).count() > 0, "No non-promo revenue found"
assert result_4.select('i_category').distinct().count() == cnt, "Duplicate categories found"
print(f"Problem 4 passed ✓  ({cnt} rows)")


## Problem 5 — Top Brands per Quarter by Revenue

Join `store_sales` + `date_dim` + `item`. Compute `total_net_paid` per brand per `d_quarter_name`.  
Window-rank brands within each quarter by revenue descending. Keep only `brand_rank <= 5`.

**Expected columns:** `d_quarter_name`, `i_brand`, `total_revenue`, `brand_rank`


In [ ]:
# Problem 5 — write your solution here
# Assign result to: result_5

result_5 = None  # replace this


In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
for expected in ['d_quarter_name', 'i_brand', 'total_revenue', 'brand_rank']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_5.count()
assert cnt > 0, f"Got 0 rows"
assert result_5.filter(F.col('brand_rank') > 5).count() == 0, "Ranks > 5 found — filter broken"
assert result_5.filter(F.col('brand_rank') == 1).count() >= 1, "No rank-1 brand found"
print(f"Problem 5 passed ✓  ({cnt} rows)")


## Problem 6 — Weekend vs Weekday Spending by Category

Join `store_sales` + `date_dim` + `item`. Compare revenue and `avg_net_paid` on weekends (`d_weekend='Y'`)  
vs weekdays per `i_category`. Produce an `is_weekend` boolean flag.

**Expected columns:** `i_category`, `is_weekend`, `total_revenue`, `avg_net_paid`, `transaction_count`


In [ ]:
# Problem 6 — write your solution here
# Assign result to: result_6

result_6 = None  # replace this


In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
for expected in ['i_category', 'is_weekend', 'total_revenue', 'avg_net_paid', 'transaction_count']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_6.count()
assert cnt > 0, f"Got 0 rows"
assert result_6.filter(F.col('is_weekend') == True).count() > 0, "No weekend rows found"
assert result_6.filter(F.col('is_weekend') == False).count() > 0, "No weekday rows found"
print(f"Problem 6 passed ✓  ({cnt} rows)")


## Problem 7 — Customer Age Segment Analysis

Join `store_sales` + `customer` on `ss_customer_sk = c_customer_sk`. Compute `age = 2003 - c_birth_year`.  
Bucket into: `Under 30`, `30-45`, `45-60`, `Over 60`.  
Per bucket: `customer_count` (distinct), `total_spend`, `avg_spend`.

**Expected columns:** `age_bucket`, `customer_count`, `total_spend`, `avg_spend`


In [ ]:
# Problem 7 — write your solution here
# Assign result to: result_7

result_7 = None  # replace this


In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
for expected in ['age_bucket', 'customer_count', 'total_spend', 'avg_spend']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_7.count()
assert cnt > 0, f"Got 0 rows"
buckets = [r['age_bucket'] for r in result_7.select('age_bucket').collect()]
assert any('30' in str(b) for b in buckets), "No 30-45 bucket found"
assert result_7.filter(F.col('customer_count') > 0).count() == cnt, "customer_count must be > 0"
print(f"Problem 7 passed ✓  ({cnt} rows)")


## Problem 8 — Loss-Making Store-Item Combinations

Join `store_sales` + `store` + `item`. Find store-item pairs where total `ss_net_profit < 0`.  
Sort by `total_loss` ascending (largest losses first).

**Expected columns:** `s_store_name`, `i_product_name`, `i_category`, `total_loss`, `transaction_count`


In [ ]:
# Problem 8 — write your solution here
# Assign result to: result_8

result_8 = None  # replace this


In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────
assert result_8 is not None, "result_8 is None"
assert hasattr(result_8, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
for expected in ['s_store_name', 'i_product_name', 'i_category', 'total_loss', 'transaction_count']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_8.count()
assert cnt > 0, f"Got 0 rows"
assert result_8.filter(F.col('total_loss') >= 0).count() == 0, "Non-negative total_loss found — filter broken"
assert result_8.filter(F.col('transaction_count') <= 0).count() == 0, "transaction_count must be > 0"
print(f"Problem 8 passed ✓  ({cnt} rows)")
